# AML Anomaly Detection (Rules + Z-Score)\n\nBusiness logic:\n1. Apply deterministic AML rules (high amount, channel/velocity behavior).\n2. Compute merchant-level z-score outliers from transaction amounts.\n3. Generate anomaly table for case triage and investigation.

In [ ]:
from pyspark.sql import functions as F\nfrom pyspark.sql.window import Window\n\ntx = spark.table(\"Retail_Transactions\").alias(\"t\")\nacct = spark.table(\"Retail_Accounts\").select(\"AccountID\", \"CustomerID\").alias(\"a\")\n\ntx_enriched = tx.join(acct, on=\"AccountID\", how=\"left\")\n\n# Rule 1: large transaction amount threshold.\nrule_large_amt = F.abs(F.col(\"Amount\")) >= 7500\n\n# Rule 2: high-frequency burst by customer within a 1-hour window.\ntx_enriched = tx_enriched.withColumn(\"HourBucket\", F.date_trunc(\"hour\", F.col(\"Timestamp\")))\nw_hour = Window.partitionBy(\"CustomerID\", \"HourBucket\")\ntx_freq = tx_enriched.withColumn(\"HourTxCount\", F.count(\"TxID\").over(w_hour))\nrule_velocity = F.col(\"HourTxCount\") >= 10\n\n# Z-score baseline by merchant category.\nstats = (\n    tx_freq.groupBy(\"MerchantCategory\")\n    .agg(F.avg(F.abs(\"Amount\")).alias(\"AvgAbsAmount\"), F.stddev_pop(F.abs(\"Amount\")).alias(\"StdAbsAmount\"))\n)\n\nanomalies = (\n    tx_freq.join(stats, on=\"MerchantCategory\", how=\"left\")\n    .withColumn(\"ZScore\", F.when(F.col(\"StdAbsAmount\").isNull() | (F.col(\"StdAbsAmount\") == 0), F.lit(0.0)).otherwise((F.abs(F.col(\"Amount\")) - F.col(\"AvgAbsAmount\")) / F.col(\"StdAbsAmount\")))\n    .withColumn(\"RuleLargeAmount\", rule_large_amt)\n    .withColumn(\"RuleVelocity\", rule_velocity)\n    .withColumn(\"RuleZOutlier\", F.abs(F.col(\"ZScore\")) >= 3)\n    .withColumn(\"AnomalyFlag\", F.col(\"RuleLargeAmount\") | F.col(\"RuleVelocity\") | F.col(\"RuleZOutlier\"))\n    .withColumn(\"RiskScore\", F.round(F.when(F.col(\"RuleLargeAmount\"), 35).otherwise(0) + F.when(F.col(\"RuleVelocity\"), 30).otherwise(0) + F.when(F.col(\"RuleZOutlier\"), 35).otherwise(0), 2))\n    .withColumn(\"Reason\", F.concat_ws(\"; \", F.when(F.col(\"RuleLargeAmount\"), F.lit(\"Large transaction amount\")), F.when(F.col(\"RuleVelocity\"), F.lit(\"High transaction velocity\")), F.when(F.col(\"RuleZOutlier\"), F.lit(\"Z-score outlier\"))))\n    .withColumn(\"DetectedAt\", F.current_timestamp())\n    .filter(F.col(\"AnomalyFlag\") == True)\n    .select(\"DetectedAt\", \"TxID\", \"CustomerID\", \"AccountID\", \"Amount\", \"Channel\", \"MerchantCategory\", \"HourTxCount\", \"ZScore\", \"RiskScore\", \"Reason\")\n)\n\n(\n    anomalies\n    .write.format(\"delta\")\n    .mode(\"overwrite\")\n    .option(\"overwriteSchema\", \"true\")\n    .save(\"Files/AML/DetectedAnomalies\")\n)\nspark.sql(\"CREATE TABLE IF NOT EXISTS AML_DetectedAnomalies USING DELTA LOCATION 'Files/AML/DetectedAnomalies'\")\n\ndisplay(anomalies.orderBy(F.desc(\"RiskScore\"), F.desc(\"Amount\")).limit(100))\n